# Validation Tests — Part 2

This notebook tests generated-name validation, connector-specific checks, and happy path scenarios.

| # | Scenario | Expected Result |
|---|----------|-----------------|
| 8 | Conflicting schedule in same pipeline group | `ValidationError` |
| 9 | Conflicting pause_status in same pipeline group | `ValidationError` |
| 10 | Conflicting connection_name in same pipeline group (SaaS) | `ValidationError` |
| 11 | Cross-project pipeline group name collision | `ValidationError` |
| 12 | Resource name exceeding 128 characters | `ValidationError` |
| 13 | Mixed subgroup usage within a prefix | `ValidationError` |
| 14 | Missing project_name | `ConfigurationError` |
| 15 | Empty DataFrame | `ConfigurationError` |
| 16 | Invalid SCD type for connector | Warning, proceeds successfully |
| 17 | Happy path: SQL Server basic | Success |
| 18 | Happy path: Salesforce with column filtering | Success |
| 19 | Happy path: Load balancing splits | Success |
| 20 | Happy path: Group-based config overrides | Success |

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%pip install pyyaml

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

username = w.current_user.me().user_name
workspace_host = w.config.host

WORKSPACE_HOST = workspace_host
ROOT_PATH = f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}'

In [ ]:
import sys
import os
import logging
import pandas as pd

sys.path.append(os.path.abspath('../../src'))

from tapworks.core.runner import run_pipeline_generation

# Enable logging to see warnings and info messages
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('tapworks')
logger.setLevel(logging.INFO)

targets = {
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
}

---
## Validation Point 2: Generated Names & Group Consistency
---

## Test 8: Conflicting Schedule in Same Pipeline Group

All 3 rows share `prefix=sales, subgroup=01` but row 2 has `*/30 * * * *` while others have `*/15 * * * *`.

**Expected:** `ValidationError` — conflicting schedule values in pipeline group.

In [ ]:
pd.read_csv('08_conflicting_schedule/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='08_conflicting_schedule/pipeline_config.csv',
        output_dir='08_conflicting_schedule/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 9: Conflicting pause_status in Same Pipeline Group

All 3 rows share `prefix=sales, subgroup=01` but row 2 has `UNPAUSED` while others have `PAUSED`.

**Expected:** `ValidationError` — conflicting pause_status values in pipeline group.

In [ ]:
pd.read_csv('09_conflicting_pause_status/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='09_conflicting_pause_status/pipeline_config.csv',
        output_dir='09_conflicting_pause_status/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 10: Conflicting connection_name in Same Pipeline Group (SaaS)

Uses Salesforce connector. Row 2 (Contact) has `sfdc_connection_b` while others have `sfdc_connection_a`.

**Expected:** `ValidationError` — conflicting connection_name in pipeline group.

In [ ]:
pd.read_csv('10_conflicting_connection_name/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='salesforce',
        input_source='10_conflicting_connection_name/pipeline_config.csv',
        output_dir='10_conflicting_connection_name/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 11: Cross-Project Pipeline Group Name Collision

Two projects (`project_alpha`, `project_beta`) both use `prefix=sales, subgroup=01`, producing the same pipeline group name.

**Expected:** `ValidationError` — pipeline group appears in multiple projects.

In [ ]:
pd.read_csv('11_cross_project_collision/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='11_cross_project_collision/pipeline_config.csv',
        output_dir='11_cross_project_collision/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 12: Resource Name Exceeding 128 Characters

Uses a very long prefix so that the generated resource name (`pipeline_<prefix>_01_g01_p01`) exceeds 128 characters.

**Expected:** `ValidationError` — exceeds maximum resource name length.

In [ ]:
pd.read_csv('12_resource_name_too_long/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='12_resource_name_too_long/pipeline_config.csv',
        output_dir='12_resource_name_too_long/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

---
## Connector-Specific Validation
---

## Test 13: Mixed Subgroup Usage Within a Prefix

Prefix `sales` has row 1 with `subgroup=01`, row 2 with empty subgroup, and row 3 with `subgroup=02`.

**Expected:** `ValidationError` — mixed subgroup usage; all tables in a prefix must have explicit subgroups when any do.

In [ ]:
pd.read_csv('13_mixed_subgroup_usage/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='13_mixed_subgroup_usage/pipeline_config.csv',
        output_dir='13_mixed_subgroup_usage/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 14: Missing project_name

CSV has no `project_name` column and no default is provided.

**Expected:** `ConfigurationError` — missing required field: project_name.

In [ ]:
pd.read_csv('14_missing_project_name/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='14_missing_project_name/pipeline_config.csv',
        output_dir='14_missing_project_name/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 15: Empty DataFrame

CSV has headers only, no data rows.

**Expected:** `ConfigurationError` — input DataFrame is empty.

In [ ]:
pd.read_csv('15_empty_dataframe/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='15_empty_dataframe/pipeline_config.csv',
        output_dir='15_empty_dataframe/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 16: Invalid SCD Type for Connector

Rows 1-2 use invalid SCD types (`SCD_TYPE_3`, `INVALID`). Row 3 uses valid `SCD_TYPE_1`.

**Expected:** Warnings logged for invalid SCD types (ignored), pipeline generation **succeeds**.

In [ ]:
pd.read_csv('16_invalid_scd_type/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='16_invalid_scd_type/pipeline_config.csv',
        output_dir='16_invalid_scd_type/deployment',
        targets=targets,
    )
    print(f'SUCCESS: Processed {len(result)} rows (check logs above for SCD type warnings)')
    print(f'Pipeline groups: {result["pipeline_group"].nunique()}')
except Exception as e:
    print(f'UNEXPECTED ERROR ({type(e).__name__}): {e}')

---
## Happy Path Scenarios
---

## Test 17: Happy Path — SQL Server Basic

Valid SQL Server config with 3 tables. Verifies YAML output is generated correctly.

**Expected:** Success. Check pipeline groups and gateway assignments.

In [ ]:
pd.read_csv('17_happy_path_sql_server/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='17_happy_path_sql_server/pipeline_config.csv',
        output_dir='17_happy_path_sql_server/deployment',
        targets=targets,
    )
    print(f'SUCCESS: Processed {len(result)} rows')
    print(f'Pipeline groups: {result["pipeline_group"].unique().tolist()}')
    print(f'Gateways: {result["gateway"].unique().tolist()}')
    display(result[['source_table_name', 'target_table_name', 'pipeline_group', 'gateway']])
except Exception as e:
    print(f'UNEXPECTED ERROR ({type(e).__name__}): {e}')

## Test 18: Happy Path — Salesforce with Column Filtering

Valid Salesforce config: Account uses `include_columns`, Lead uses `exclude_columns`, others have no filtering.

**Expected:** Success. No row uses both include and exclude.

In [ ]:
pd.read_csv('18_happy_path_salesforce_columns/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='salesforce',
        input_source='18_happy_path_salesforce_columns/pipeline_config.csv',
        output_dir='18_happy_path_salesforce_columns/deployment',
        targets=targets,
    )
    print(f'SUCCESS: Processed {len(result)} rows')
    print(f'Pipeline groups: {result["pipeline_group"].unique().tolist()}')
    display(result[['source_table_name', 'target_table_name', 'pipeline_group', 'include_columns', 'exclude_columns']])
except Exception as e:
    print(f'UNEXPECTED ERROR ({type(e).__name__}): {e}')

## Test 19: Happy Path — Load Balancing Splits

18 tables with `max_tables_per_gateway=10` and `max_tables_per_pipeline=5`. Should split into multiple gateways and pipelines.

**Expected:** Success. 2 gateways, 4 pipelines (gateway 1: 2 pipelines of 5, gateway 2: 2 pipelines of 5+3).

In [ ]:
pd.read_csv('19_load_balancing_splits/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='19_load_balancing_splits/pipeline_config.csv',
        output_dir='19_load_balancing_splits/deployment',
        targets=targets,
        max_tables_per_gateway=10,
        max_tables_per_pipeline=5,
    )
    print(f'SUCCESS: Processed {len(result)} rows')
    print(f'Gateways: {result["gateway"].unique().tolist()}')
    print(f'Pipeline groups: {result["pipeline_group"].unique().tolist()}')
    print(f'\nTables per gateway:')
    print(result.groupby('gateway')['source_table_name'].count())
    print(f'\nTables per pipeline:')
    print(result.groupby('pipeline_group')['source_table_name'].count())
except Exception as e:
    print(f'UNEXPECTED ERROR ({type(e).__name__}): {e}')

## Test 20: Happy Path — Group-Based Config Overrides

CSV has empty schedules. Uses `default_values` with per-prefix schedules: `sales` gets `*/15 * * * *`, `marketing` gets `0 * * * *`.

**Expected:** Success. Each prefix gets its own schedule applied via group-based defaults.

In [ ]:
pd.read_csv('20_group_based_overrides/pipeline_config.csv')

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='20_group_based_overrides/pipeline_config.csv',
        output_dir='20_group_based_overrides/deployment',
        targets=targets,
        default_values={
            'sales': {'schedule': '*/15 * * * *'},
            'marketing': {'schedule': '0 * * * *'},
        },
    )
    print(f'SUCCESS: Processed {len(result)} rows')
    print(f'\nSchedule per prefix:')
    display(result[['source_table_name', 'prefix', 'pipeline_group', 'schedule']])
except Exception as e:
    print(f'UNEXPECTED ERROR ({type(e).__name__}): {e}')